In [ ]:
# Purpose - makes fields/spectra movie (newer)

findex = 23 #63 #6 #25

In [ ]:
# Imports and Folder Selection

import sys

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun
#from FieldLineHelper import FieldLineHelper

plt.close('all')
%matplotlib widget
set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
boilerplate = run.boilerplate #run.boilerplate + f"tstep = {run.steps[0]}"

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}

my_steps = tstep_choices["all"]

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields")

In [ ]:
def compute_Az(roi):    
    roi['az'] = (np.cumsum(roi.bx, axis=0) - np.cumsum(roi.by, axis=1)) * run.istep #properly should multiply this by the spacing

In [ ]:
from scipy.signal import find_peaks

roi = cs = fq_cut(d.fields, 119, [0, run.xlim, 0, run.ylim])
compute_Az(roi)

midline = fq_ycut(roi, run.xlim / 2)

NPASSES = 5
AZ_THRESHOLD = 0.08

az = midline.az

plt.figure()

plt.plot(az)

# Find peaks (local maxima) in the az data
peaks, _ = find_peaks(az)

def level_bounds(maxima_y, level):
    crossings = np.where(np.diff(np.signbit(az - level)))[0]

    lower_crossings = crossings[crossings < maxima_y]
    upper_crossings = crossings[crossings > maxima_y]
    
    if lower_crossings.size > 0 and upper_crossings.size > 0:
        return [np.max(lower_crossings), np.min(upper_crossings)]
    else:
        return None


def find_plasmoid(maxima_y):
    count = 0
    left = az.min()
    right = az[maxima_y]
    best_bounds = None

    while NPASSES > count: 
        possible_level = (left + right) / 2

        bounds = level_bounds(maxima_y, possible_level)

        if bounds is None:
            left = possible_level
        else:  
            right = possible_level
            best_bounds = bounds

        count += 1

    if best_bounds is None or (az[maxima_y] - az[best_bounds[0]] < AZ_THRESHOLD):
        return None

    return (az.coords['y'] >= az.coords['y'][best_bounds[0]]) & (az.coords['y'] <= az.coords['y'][best_bounds[1]])

# Find points where az passes from below to above 5 or vice-versa
threshold = 5

# Plot the peaks
plt.plot(peaks, midline.az[peaks], "x")

plasmoid_masks = []

for peak in peaks:
    mask = find_plasmoid(peak)
    if mask is not None:
        plasmoid_masks.append(mask)

if plasmoid_masks:
    plasmoid_mask = np.logical_or.reduce(plasmoid_masks)
    for mask in plasmoid_masks:
        bounds = np.where(mask)[0]
        plt.axvspan(bounds[0], bounds[-1], color='blue', alpha=0.3)
else:
    plasmoid_mask = None

# Print the list of local maxima
print("Local maxima of az:", peaks)

plt.xlabel('y')
plt.ylabel('vely')
plt.show()

In [ ]:
filename = f'{findex}_timemask.npy'
gen_new = False

try:
    timemask = np.load(filename)
    print(f"Loaded timemask from {filename}")
except FileNotFoundError:
    print(f"{filename} not found, generating new timemask")
    gen_new = True

if gen_new:
    n = len(my_steps)
    ylen = len(az.coords['y'])

    timemask = np.zeros((n, ylen), dtype=bool)

    for i in range(n):
        roi = fq_cut(d.fields, i, [0, run.xlim, 0, run.ylim])
        compute_Az(roi)
        midline = fq_ycut(roi, run.xlim / 2)
        az = midline.az
        peaks, _ = find_peaks(az)

        plasmoid_masks = []

        for peak in peaks:
            mask = find_plasmoid(peak)
            if mask is not None:
                plasmoid_masks.append(mask)
        if plasmoid_masks:
            plasmoid_mask = np.logical_or.reduce(plasmoid_masks)
            timemask[i, :] = plasmoid_mask
        else:
            timemask[i, :] = False

In [ ]:


plt.figure()
plt.imshow(timemask, aspect='auto', origin='lower', cmap='viridis', extent = [0, run.ylim, 0, (my_steps[-1] - my_steps[0]) * run.gyrofreq])
plt.xlabel('$y$ [$c / \omega_p$]')
plt.ylabel('$\Omega t$')
plt.title(boilerplate)

save_polished_fig(f'{findex}_timemask')

plt.show()

In [ ]:
np.save(filename, timemask)